# CinnaSense AI: Model 01 two-class visual prototype

Use Colab **Runtime > Change runtime type > GPU**. Upload the `ml-module02` folder and the condition-photo folder to Google Drive. Review the manifest locally first. Do not upload database/device secrets. This notebook does not train from commercial-grade labels or YOLO Quill labels.

Training is intentionally blocked until conflicting labels, exact duplicates and physical sample IDs are reviewed. Folder labels alone are not mold confirmation. No accuracy or trained artifact has been fabricated.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
# Edit these paths to your uploaded folders.
PROJECT = Path('/content/drive/MyDrive/CinnaSense/ml-module02')
DATASET = Path('/content/drive/MyDrive/CinnaSense/OneDrive_1_10-20-2026')
MANIFEST = PROJECT / 'data/reviewed_manifest.csv'
assert PROJECT.joinpath('pyproject.toml').exists(), 'Upload the ml-module02 source folder.'
assert DATASET.exists(), 'Upload the actual photos separately; paths in the manifest are relative.'


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT) + '[training]'])
import tensorflow as tf
print('TensorFlow:', tf.__version__, 'GPUs:', tf.config.list_physical_devices('GPU'))


## Validate the reviewed manifest before training

If you have not reviewed labels and sample IDs, stop here. Use `python -m cinnasense_quality.dataset --root <photos> --output <new-audit-folder>` to create a review template. Do not mark all rows verified without inspection. Remove redundant duplicate rows in the manifest; original photos need not be deleted. Keep all photos of a physical sample together even across dates.


In [ ]:
from cinnasense_quality.dataset import load_reviewed, split_groups
rows = load_reviewed(MANIFEST, DATASET)
splits = split_groups(rows)
from collections import Counter
for name, subset in splits.items():
    print(name, 'images:', len(subset), 'physical samples:', len({r['sample_id'] for r in subset}), Counter(r['label'] for r in subset))


## Train and evaluate once on the held-out test subset

The pipeline uses ImageNet MobileNetV2, modest pose augmentation, embedded [-1,1] scaling, a classification head, then optional fine tuning. Validation chooses the weights; held-out test images are evaluated only after model selection. Do not repeatedly tune against the test report. Keep the split CSVs with your experiment. Small-sample scores and camera-domain shifts limit conclusions.


In [ ]:
from cinnasense_quality.train import train
from datetime import datetime, timezone
RUN = PROJECT / 'artifacts' / ('condition-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
report = train(DATASET, MANIFEST, RUN, epochs=12, fine_tune_epochs=5)
print('Artifact folder:', RUN)
print(report)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
import numpy as np
ConfusionMatrixDisplay(np.asarray(report['confusion_matrix']), display_labels=report['classes']).plot(cmap='Greens', values_format='d')
plt.title('Held-out physical-sample test set'); plt.show()


## Deploy only the generated, evaluated artifact

Download the archive and extract it under `ml-module02/artifacts/<run-name>` locally. Set `MODULE02_IMAGE_MODEL_DIR` to that absolute folder, install the API dependencies in a separate Python environment, and start port 8002. Express keeps using port 5000; the price service keeps port 8000. Do not change labels or preprocessing after export. Model scores are uncalibrated classification scores; Suspected Mold is a visual suspicion, not a laboratory result.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/' + RUN.name, 'zip', RUN)
files.download(archive)
